# Multi-Head Stance Model — Turn-Level Analysis (v2)

Adapts the chunk-level multi-head DeBERTa analysis to **speaker turns** as the unit of analysis.
Three LLMs: Llama 3.3, DeepSeek V3, Mistral Large (Qwen to be added when available).

## Structure
1. **Dictionary baseline** — hawk/dove word count → predict disagreement (AUC floor)
2. **TF-IDF + SHAP** — statistical text features → predict disagreement
3. **Multi-Head DeBERTa** — shared encoder + one linear head per LLM, trained jointly
4. **Gradient attribution** — which tokens drive each head's prediction?
5. **Head weight analysis** — how similar are the learned linear projections?
6. **Swap analysis** — encoding vs. decision-rule decomposition
7. **Linear probe** — AUC comparison across all three methods + risk timeline

**Upload** `turn_predictions_llama33.csv`, `turn_predictions_deepseekv3.csv`,
`turn_predictions_mistrallarge_or.csv` to `/content/` before running.

**Runtime:** ~30–40 min on T4 GPU for DeBERTa training.

In [ ]:
!pip install -q transformers accelerate scikit-learn shap scipy matplotlib seaborn

In [ ]:
import os, re, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, DebertaV2Model
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, ConfusionMatrixDisplay,
                             accuracy_score, f1_score)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## Data Loading

Upload the three CSVs to `/content/`.

`split = True` when 1 or 2 of the 3 models classify a turn as directional,
while the remaining model(s) classify it as neutral — partial disagreement on
whether the turn takes a monetary policy stance at all.

*Note: Qwen 2.5 72B will be added to MODELS when available.*

In [ ]:
MODELS = {
    "llama33":         "Llama 3.3",
    "deepseekv3":      "DeepSeek V3",
    "mistrallarge_or": "Mistral Large",
}

LABEL2IDX = {"dovish":0,"mostly dovish":1,"neutral":2,"mostly hawkish":3,"hawkish":4}
IDX2LABEL  = {v:k for k,v in LABEL2IDX.items()}
STANCED    = {"dovish","mostly dovish","mostly hawkish","hawkish"}
SCORE_MAP  = {"dovish":-2,"mostly dovish":-1,"neutral":0,"mostly hawkish":1,"hawkish":2}
MODEL_NAME = "microsoft/deberta-v3-base"

dfs = {}
for key in MODELS:
    path = f"/content/turn_predictions_{key}.csv"
    df   = pd.read_csv(path)
    df   = df[df["label"].isin(LABEL2IDX)].copy()
    dfs[key] = df
    print(f"{key}: {len(df)} turns")

base_cols  = ["turn_uid","bank","date","speaker","text"]
first_key  = list(MODELS.keys())[0]
turns_wide = dfs[first_key][base_cols].copy()
loaded     = []

for key, df in dfs.items():
    sub = df[["turn_uid","label"]].rename(columns={"label":f"label_{key}"})
    turns_wide = turns_wide.merge(sub, on="turn_uid", how="inner")
    turns_wide[f"score_{key}"] = turns_wide[f"label_{key}"].map(SCORE_MAP)
    loaded.append(key)

for key in loaded:
    turns_wide[f"stanced_{key}"] = turns_wide[f"label_{key}"].isin(STANCED).astype(int)

turns_wide["n_stanced"] = turns_wide[[f"stanced_{k}" for k in loaded]].sum(axis=1)
turns_wide["split"]     = turns_wide["n_stanced"].between(1, len(loaded)-1)

rng  = np.random.RandomState(SEED)
uids = turns_wide["turn_uid"].unique()
rng.shuffle(uids)
n    = len(uids)
n_tr = int(0.70*n); n_va = int(0.15*n)
split_map = {uid:"train" for uid in uids[:n_tr]}
split_map.update({uid:"val"  for uid in uids[n_tr:n_tr+n_va]})
split_map.update({uid:"test" for uid in uids[n_tr+n_va:]})
turns_wide["split_set"] = turns_wide["turn_uid"].map(split_map)

print(f"\nTotal turns : {len(turns_wide)}")
print(f"Split turns : {turns_wide['split'].sum()} ({turns_wide['split'].mean():.1%})")
print(f"Train: {(turns_wide['split_set']=='train').sum()}  "
      f"Val: {(turns_wide['split_set']=='val').sum()}  "
      f"Test: {(turns_wide['split_set']=='test').sum()}")

## Baseline 1 — Dictionary (AUC Floor)

Count hawk/dove words per turn; net score predicts disagreement.
This is the simplest possible text feature. If TF-IDF and DeBERTa cannot beat this,
the more complex methods add nothing.

In [ ]:
HAWK_WORDS = {
    "hike","hikes","hiking","tighten","tightening","tightened",
    "raise","raises","raising","restrictive","restriction",
    "inflation","inflationary","overshoot","overheating",
    "hawkish","normalisation","normalization","unwind",
}
DOVE_WORDS = {
    "cut","cuts","cutting","accommodation","accommodative",
    "stimulus","easing","ease","support","lower","lowering",
    "dovish","expansionary","unconventional","qe","purchase",
    "below","undershoot",
}

def dict_score(text):
    tokens = re.findall(r"[a-z]+", str(text).lower())
    h = sum(1 for t in tokens if t in HAWK_WORDS)
    d = sum(1 for t in tokens if t in DOVE_WORDS)
    return h - d

turns_wide["dict_score"] = turns_wide["text"].apply(dict_score)

tr_mask = turns_wide["split_set"] == "train"
te_mask = turns_wide["split_set"] == "test"
y_tr = turns_wide.loc[tr_mask,"split"].astype(int).values
y_te = turns_wide.loc[te_mask,"split"].astype(int).values

X_dict_tr = turns_wide.loc[tr_mask,"dict_score"].values.reshape(-1,1)
X_dict_te = turns_wide.loc[te_mask,"dict_score"].values.reshape(-1,1)

lr_dict  = LogisticRegression(class_weight="balanced")
lr_dict.fit(X_dict_tr, y_tr)
auc_dict = roc_auc_score(y_te, lr_dict.predict_proba(X_dict_te)[:,1])
print(f"Dictionary AUC: {auc_dict:.3f}  (expected ~0.52–0.58)")

## Baseline 2 — TF-IDF + SHAP

Statistical text features weighted by corpus frequency.
SHAP shows which words drive prediction of split vs. consensus.

Compare top SHAP words here against DeBERTa gradient attribution results later:
if the same vocabulary appears in both, the finding is robust to method.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import shap

train_texts = turns_wide.loc[tr_mask,"text"].tolist()
test_texts  = turns_wide.loc[te_mask,"text"].tolist()

tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2),
                        min_df=3, sublinear_tf=True)
X_tr_tfidf = tfidf.fit_transform(train_texts)
X_te_tfidf = tfidf.transform(test_texts)

lr_tfidf = LogisticRegression(max_iter=1000, class_weight="balanced", C=1.0)
lr_tfidf.fit(X_tr_tfidf, y_tr)
auc_tfidf = roc_auc_score(y_te, lr_tfidf.predict_proba(X_te_tfidf)[:,1])
print(f"TF-IDF AUC : {auc_tfidf:.3f}")
print(f"Dictionary : {auc_dict:.3f}")

explainer   = shap.LinearExplainer(lr_tfidf, X_tr_tfidf,
                                    feature_perturbation="interventional")
shap_vals   = explainer.shap_values(X_te_tfidf)
feat_names  = tfidf.get_feature_names_out()
mean_shap   = np.abs(shap_vals).mean(axis=0)
top_idx     = np.argsort(mean_shap)[::-1][:25]

fig, ax = plt.subplots(figsize=(10,6))
ax.barh(range(25), mean_shap[top_idx][::-1], color="#4C72B0")
ax.set_yticks(range(25))
ax.set_yticklabels([feat_names[i] for i in top_idx[::-1]], fontsize=9)
ax.set_xlabel("Mean |SHAP value|")
ax.set_title("TF-IDF Top 25 Features Predicting LLM Disagreement")
plt.tight_layout()
plt.savefig("tfidf_shap_disagreement.png", dpi=130, bbox_inches="tight")
plt.show()

## Multi-Head DeBERTa

```
turn text -> DeBERTa-v3-base -> [CLS] 768-dim
                                     |
                  ┌──────────────────┼──────────────────┐
             head_llama33     head_deepseekv3    head_mistral
             Linear(768,5)    Linear(768,5)     Linear(768,5)
```

All heads trained simultaneously; shared encoder satisfies all three labeling functions.

**Note:** Turns are truncated to 512 tokens. Gradient attribution covers first ~400 words only.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class MultiHeadDataset(Dataset):
    def __init__(self, df, model_keys, max_length=512):
        self.df = df.reset_index(drop=True)
        self.model_keys = model_keys
        self.max_length = max_length

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        enc = tokenizer(
            str(row["text"]),
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        item = {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "turn_uid":       row["turn_uid"],
            "is_split":       int(row["split"]),
        }
        for key in self.model_keys:
            item[f"label_{key}"] = LABEL2IDX.get(row[f"label_{key}"], LABEL2IDX["neutral"])
        return item

BATCH = 8
train_df = turns_wide[turns_wide["split_set"]=="train"]
val_df   = turns_wide[turns_wide["split_set"]=="val"]
test_df  = turns_wide[turns_wide["split_set"]=="test"]

train_loader = DataLoader(MultiHeadDataset(train_df, loaded), batch_size=BATCH,
                          shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(MultiHeadDataset(val_df,   loaded), batch_size=BATCH,
                          shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(MultiHeadDataset(test_df,  loaded), batch_size=BATCH,
                          shuffle=False, num_workers=2, pin_memory=True)
print(f"Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}")

In [ ]:
class MultiHeadStanceModel(nn.Module):
    def __init__(self, encoder_name, model_keys, num_labels=5):
        super().__init__()
        self.encoder = DebertaV2Model.from_pretrained(encoder_name)
        self.heads   = nn.ModuleDict({
            key: nn.Linear(768, num_labels) for key in model_keys
        })

    def forward(self, input_ids, attention_mask):
        enc = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = enc.last_hidden_state[:, 0, :].float()
        logits = {key: head(cls) for key, head in self.heads.items()}
        return logits, cls

model = MultiHeadStanceModel(MODEL_NAME, loaded).to(device)
print(f"Encoder params: {sum(p.numel() for p in model.encoder.parameters()):,}")
print(f"Heads: {list(model.heads.keys())}")

## Training

- Epoch 1: encoder frozen, heads only (lr=1e-3)
- Epochs 2–3: full fine-tune, encoder at lr=2e-5
- Gradient accumulation 4 steps → effective batch 32

In [ ]:
# ── Resume from checkpoint (set SKIP_TRAINING=True to load, False to train) ──
SKIP_TRAINING   = True
CKPT_FROM_DRIVE = True

if SKIP_TRAINING:
    if CKPT_FROM_DRIVE:
        from google.colab import drive
        drive.mount("/drive", force_remount=False)
        CKPT_PATH = "/drive/MyDrive/central_bank_spillovers/multihead_turns/model_turns_checkpoint.pt"
    else:
        from google.colab import files as colab_files
        print("Upload model_turns_checkpoint.pt")
        up = colab_files.upload()
        CKPT_PATH = list(up.keys())[0]

    ckpt = torch.load(CKPT_PATH, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    model = model.float().to(device)
    history       = ckpt.get("history", [])
    best_val_loss = ckpt.get("best_val_loss", float("inf"))
    print(f"Loaded: {CKPT_PATH}")
else:
    print("SKIP_TRAINING=False — run training cell next.")

In [ ]:
if not SKIP_TRAINING:
    EPOCHS = 3; GRAD_ACC = 4
    model  = model.float()
    crit   = nn.CrossEntropyLoss()

    def make_opt(m, enc_lr=2e-5, head_lr=1e-3):
        return torch.optim.AdamW([
            {"params": m.encoder.parameters(), "lr": enc_lr, "weight_decay": 0.01},
            {"params": m.heads.parameters(),   "lr": head_lr, "weight_decay": 0.01},
        ])

    @torch.no_grad()
    def evaluate(loader):
        model.eval()
        preds = {k:[] for k in loaded}; trues = {k:[] for k in loaded}; tot = 0.0
        for b in loader:
            ids = b["input_ids"].to(device); mask = b["attention_mask"].to(device)
            lg, _ = model(ids, mask)
            loss = sum(crit(lg[k], b[f"label_{k}"].to(device)) for k in loaded)
            tot += loss.item()
            for k in loaded:
                preds[k].extend(lg[k].argmax(-1).cpu().tolist())
                trues[k].extend(b[f"label_{k}"].tolist())
        accs = {k: np.mean(np.array(preds[k])==np.array(trues[k])) for k in loaded}
        return tot/len(loader), accs

    history = []; best_val_loss = float("inf")

    for epoch in range(1, EPOCHS+1):
        for p in model.encoder.parameters(): p.requires_grad = (epoch > 1)
        opt = make_opt(model, enc_lr=(2e-5 if epoch>1 else 0))
        print(f"Epoch {epoch}: {'frozen' if epoch==1 else 'full fine-tune'}")
        model.train(); opt.zero_grad(); run_loss = 0.0
        for step, b in enumerate(train_loader, 1):
            ids = b["input_ids"].to(device); mask = b["attention_mask"].to(device)
            lg, _ = model(ids, mask)
            loss  = sum(crit(lg[k], b[f"label_{k}"].to(device)) for k in loaded) / GRAD_ACC
            loss.backward(); run_loss += loss.item() * GRAD_ACC
            if step % GRAD_ACC == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step(); opt.zero_grad()
            if step % 50 == 0:
                print(f"  step {step}/{len(train_loader)}  loss={run_loss/step:.4f}")
        vl, va = evaluate(val_loader)
        print(f"  val_loss={vl:.4f}  accs={va}")
        history.append({"epoch":epoch,"val_loss":vl,"val_accs":va})
        if vl < best_val_loss: best_val_loss = vl
    print("Training complete.")

In [ ]:
if not SKIP_TRAINING:
    from google.colab import drive as _drv
    _drv.mount("/drive", force_remount=False)
    DRIVE_DIR = "/drive/MyDrive/central_bank_spillovers/multihead_turns"
    os.makedirs(DRIVE_DIR, exist_ok=True)
    CKPT_PATH = os.path.join(DRIVE_DIR, "model_turns_checkpoint.pt")
    torch.save({"model_state_dict":model.state_dict(),"loaded_keys":loaded,
                "label2idx":LABEL2IDX,"model_name":MODEL_NAME,
                "history":history,"best_val_loss":best_val_loss}, CKPT_PATH)
    print(f"Saved: {CKPT_PATH}")

## Evaluation — Per-Head Metrics

In [ ]:
model.eval()
all_preds = {k:[] for k in loaded}; all_true = {k:[] for k in loaded}

with torch.no_grad():
    for b in test_loader:
        ids = b["input_ids"].to(device); mask = b["attention_mask"].to(device)
        lg, _ = model(ids, mask)
        for k in loaded:
            all_preds[k].extend(lg[k].argmax(-1).cpu().tolist())
            all_true[k].extend(b[f"label_{k}"].tolist())

rows = []
print("=== Test Set ===")
for k in loaded:
    yp = np.array(all_preds[k]); yt = np.array(all_true[k])
    acc = accuracy_score(yt,yp); f1 = f1_score(yt,yp,average="macro",zero_division=0)
    rows.append({"Model":MODELS[k],"Accuracy":f"{acc:.3f}","Macro-F1":f"{f1:.3f}"})
    print(f"{MODELS[k]:22s}  acc={acc:.3f}  macro-F1={f1:.3f}")
    print(classification_report(yt,yp,target_names=list(LABEL2IDX),zero_division=0))

pd.DataFrame(rows).to_csv("per_head_metrics.csv",index=False)

In [ ]:
fig, axes = plt.subplots(1,len(loaded),figsize=(5*len(loaded),4))
lnames = list(LABEL2IDX.keys())
for ax,k in zip(axes,loaded):
    cm = confusion_matrix(all_true[k],all_preds[k])
    ConfusionMatrixDisplay(cm,display_labels=lnames).plot(ax=ax,colorbar=False,xticks_rotation=45)
    ax.set_title(MODELS[k])
plt.suptitle("Confusion Matrices — Multi-Head DeBERTa (Turns)",y=1.02)
plt.tight_layout()
plt.savefig("confusion_matrices_turns.png",dpi=120,bbox_inches="tight")
plt.show()

## Gradient Attribution per Head

For each split turn in the test set: gradient of predicted class logit
w.r.t. input word embeddings (∂logit/∂embedding), L2-normed per token.

*Turns are truncated to 512 tokens — attribution covers the first ~400 words only.*

In [ ]:
STOP_WORDS = {
    "the","and","of","to","in","is","that","for","it","we","on","at","a","an",
    "be","as","by","are","this","with","have","from","our","has","was",
    "will","been","or","which","but","not","##","[CLS]","[SEP]",
}

test_split_df = turns_wide[(turns_wide["split_set"]=="test") & (turns_wide["split"])].reset_index(drop=True)
split_loader  = DataLoader(MultiHeadDataset(test_split_df,loaded,max_length=512),
                           batch_size=4,shuffle=False)

token_scores = {k:{} for k in loaded}

model.eval()
for batch in split_loader:
    ids  = batch["input_ids"].to(device)
    mask = batch["attention_mask"].to(device)
    for k in loaded:
        emb = model.encoder.embeddings.word_embeddings(ids)
        emb.retain_grad()
        out = model.encoder(inputs_embeds=emb, attention_mask=mask)
        cls = out.last_hidden_state[:,0,:].float()
        lg  = model.heads[k](cls)
        lg[range(len(lg)),lg.argmax(-1)].sum().backward(retain_graph=True)
        if emb.grad is None: model.zero_grad(); continue
        imp = emb.grad.norm(dim=-1).detach().cpu().numpy()
        for b in range(ids.shape[0]):
            toks = tokenizer.convert_ids_to_tokens(ids[b].cpu().tolist())
            for tok,sc in zip(toks,imp[b]):
                clean = tok.replace("\u2581","").replace("##","").lower()
                if clean and clean not in STOP_WORDS and len(clean)>2:
                    token_scores[k][clean] = token_scores[k].get(clean,0)+sc
        model.zero_grad()

print(f"Attribution done over {len(test_split_df)} split turns.")

In [ ]:
TOP_N  = 20
COLORS = ["#C44E52","#4C72B0","#DD8452"]

fig, axes = plt.subplots(1,len(loaded),figsize=(5*len(loaded),6))
for ax,(k,col) in zip(axes,zip(loaded,COLORS)):
    items = sorted(token_scores[k].items(),key=lambda x:x[1],reverse=True)[:TOP_N]
    toks,vals = zip(*items)
    ax.barh(range(TOP_N),list(vals)[::-1],color=col,alpha=0.85)
    ax.set_yticks(range(TOP_N))
    ax.set_yticklabels(list(toks)[::-1],fontsize=9)
    ax.set_title(MODELS[k])
    ax.set_xlabel("Cumulative gradient norm")
plt.suptitle("Gradient Attribution per Head (Split Turns)",y=1.01)
plt.tight_layout()
plt.savefig("head_gradient_attribution_turns.png",dpi=130,bbox_inches="tight")
plt.show()

## Head Weight Analysis

Each head is `Linear(768,5)`. Cosine similarity between flattened 5×768 weight matrices
measures how similar two heads' learned decision rules are.

In [ ]:
from itertools import combinations
from sklearn.decomposition import PCA

hw = {k: model.heads[k].weight.detach().cpu().numpy().flatten() for k in loaded}
pairs = list(combinations(loaded,2))

print("Head weight cosine similarities:")
for a,b in pairs:
    cos = np.dot(hw[a],hw[b])/(np.linalg.norm(hw[a])*np.linalg.norm(hw[b]))
    print(f"  {MODELS[a]} <-> {MODELS[b]}: {cos:.3f}")

W      = np.stack([hw[k] for k in loaded])
coords = PCA(n_components=2).fit_transform(W)
pca2   = PCA(n_components=2).fit(W)

fig,ax = plt.subplots(figsize=(5,4))
for i,(k,col) in enumerate(zip(loaded,COLORS)):
    ax.scatter(*coords[i],s=150,color=col,zorder=3,label=MODELS[k])
    ax.annotate(MODELS[k],coords[i],textcoords="offset points",xytext=(8,4),fontsize=9)
ax.set_xlabel(f"PC1 ({pca2.explained_variance_ratio_[0]:.0%} var)")
ax.set_ylabel(f"PC2 ({pca2.explained_variance_ratio_[1]:.0%} var)")
ax.set_title("PCA of Head Weight Matrices")
plt.tight_layout()
plt.savefig("head_weight_similarity_turns.png",dpi=130,bbox_inches="tight")
plt.show()

## Swap Analysis — Encoding vs. Decision-Rule Disagreement

Run encoder once per split turn; apply all 3 heads to the **identical** CLS vector.
- If heads agree: zero-shot disagreement was encoding-level (different LLM architectures)
- If heads still disagree: genuine decision-rule difference

With 3 models, unanimous = all 3 predict the same label.

In [ ]:
model.eval()
swap_results = []

with torch.no_grad():
    for batch in split_loader:
        ids  = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        _,cls = model(ids,mask)
        swap_preds = {k: model.heads[k](cls).argmax(-1).cpu().tolist() for k in loaded}
        for i in range(cls.shape[0]):
            labels_i = [swap_preds[k][i] for k in loaded]
            swap_results.append({"n_unique":len(set(labels_i)),
                                 **{f"pred_{k}":swap_preds[k][i] for k in loaded}})

swap_df = pd.DataFrame(swap_results)
total   = len(swap_df)
print("Swap Analysis — Consensus on Split Turns (Shared Encoder):")
for n_u,cnt in swap_df["n_unique"].value_counts().sort_index().items():
    tag = "all agree" if n_u==1 else ("all differ" if n_u==len(loaded) else "partial")
    print(f"  {n_u} unique predictions ({tag}): {cnt} ({cnt/total:.1%})")

unan = (swap_df["n_unique"]==1).mean()
print(f"\n{unan:.1%} resolve to unanimous agreement when encoding is shared")
print(f"{1-unan:.1%} are genuine decision-rule disagreements")

print("\nPairwise disagreement rates (shared CLS):")
for a,b in pairs:
    d = (swap_df[f"pred_{a}"] != swap_df[f"pred_{b}"]).mean()
    print(f"  {MODELS[a]} <-> {MODELS[b]}: {d:.1%}")

swap_df.to_csv("swap_analysis_turns.csv",index=False)

## Linear Probe — AUC Comparison + Disagreement Risk Timeline

Train logistic regression on DeBERTa CLS vectors (post fine-tuning) to predict split.

**Key table:** Dictionary AUC → TF-IDF AUC → DeBERTa CLS AUC

If DeBERTa >> TF-IDF: contextual encoding matters beyond bag-of-words.
If DeBERTa ≈ TF-IDF: the simpler method is sufficient.

In [ ]:
model.eval()
all_ds  = MultiHeadDataset(turns_wide.reset_index(drop=True),loaded,max_length=512)
all_ldr = DataLoader(all_ds,batch_size=16,shuffle=False,num_workers=2,pin_memory=True)

cls_list = []
with torch.no_grad():
    for b in all_ldr:
        ids=b["input_ids"].to(device); mask=b["attention_mask"].to(device)
        _,cls = model(ids,mask)
        cls_list.append(cls.cpu().numpy())

X_all      = np.vstack(cls_list)
y_all      = turns_wide["split"].astype(int).values
split_sets = turns_wide["split_set"].values
print(f"CLS matrix: {X_all.shape}  |  split: {y_all.sum()} / {len(y_all)}")

In [ ]:
tr_m = split_sets=="train"
te_m = split_sets=="test"

sc        = StandardScaler()
X_tr_cls  = sc.fit_transform(X_all[tr_m])
X_te_cls  = sc.transform(X_all[te_m])

probe = LogisticRegression(max_iter=1000,C=1.0,class_weight="balanced")
probe.fit(X_tr_cls,y_all[tr_m])
auc_deberta = roc_auc_score(y_all[te_m], probe.predict_proba(X_te_cls)[:,1])

print(classification_report(y_all[te_m],probe.predict(X_te_cls),
      target_names=["consensus","split"],digits=3))

print("\n=== AUC Comparison ===")
print(f"  Dictionary (hawk/dove count) : {auc_dict:.3f}")
print(f"  TF-IDF logistic regression   : {auc_tfidf:.3f}")
print(f"  DeBERTa CLS linear probe     : {auc_deberta:.3f}")

In [ ]:
all_df = turns_wide.reset_index(drop=True).copy()
all_df["disagree_risk"] = probe.predict_proba(sc.transform(X_all))[:,1]
all_df["date"] = pd.to_datetime(
    turns_wide.reset_index(drop=True)["date"].astype(str), format="%Y%m%d")

meeting_risk = (all_df.groupby(["bank","date"])["disagree_risk"]
                .mean().reset_index().sort_values(["bank","date"]))

banks = sorted(meeting_risk["bank"].unique())
fig,axes = plt.subplots(len(banks),1,figsize=(14,3.5*len(banks)),sharex=True)
if len(banks)==1: axes=[axes]

for ax,bank in zip(axes,banks):
    bdf = meeting_risk[meeting_risk["bank"]==bank]
    ax.fill_between(bdf["date"],bdf["disagree_risk"],alpha=0.25,color="#C44E52")
    ax.plot(bdf["date"],bdf["disagree_risk"],color="#C44E52",linewidth=1.2)
    ax.axhline(0.5,color="grey",linestyle="--",linewidth=0.8)
    ax.set_ylim(0,1); ax.set_ylabel("P(disagreement)"); ax.set_title(bank)

plt.xlabel("Date")
plt.suptitle(
    "Disagreement Risk Score over Time (Linear Probe on DeBERTa CLS — Turns)\n"
    "Peaks = meetings where model choice most affects estimated stance",fontsize=10)
plt.tight_layout()
plt.savefig("linear_probe_risk_timeline_turns.png",dpi=130,bbox_inches="tight")
plt.show()

all_df[["turn_uid","bank","date","split","disagree_risk"]].to_csv(
    "linear_probe_scores_turns.csv",index=False)
meeting_risk.to_csv("linear_probe_meeting_risk_turns.csv",index=False)
print("Saved.")

## Save & Download All Outputs

In [ ]:
import shutil

outputs = [
    "per_head_metrics.csv",
    "confusion_matrices_turns.png",
    "tfidf_shap_disagreement.png",
    "head_gradient_attribution_turns.png",
    "head_weight_similarity_turns.png",
    "swap_analysis_turns.csv",
    "linear_probe_risk_timeline_turns.png",
    "linear_probe_scores_turns.csv",
    "linear_probe_meeting_risk_turns.csv",
]

if not os.path.exists("/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/drive",force_remount=False)

DRIVE_DIR = "/drive/MyDrive/central_bank_spillovers/multihead_turns"
os.makedirs(DRIVE_DIR,exist_ok=True)

for fname in outputs:
    if os.path.exists(fname):
        shutil.copy(fname,os.path.join(DRIVE_DIR,fname))
        print(f"  saved: {fname}")
    else:
        print(f"  MISSING: {fname}")

from google.colab import files
for fname in outputs:
    if os.path.exists(fname): files.download(fname)